In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import os
os.chdir('C:\Kaggle_Competition\Playground\S6E4-Irrigation-Need')
import warnings
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
import seaborn as sns
import config
from src.utils import (
    read_csv_file,
    reduce_mem_usage,
    save_csv_file
)
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
from src.fe import *
import optuna
from tqdm.notebook import tqdm
from itertools import combinations
pd.set_option('display.max_columns', None)

In [4]:
train_raw = read_csv_file(r'data\raw\train.csv')
test_raw = read_csv_file(r'data\raw\test.csv')

# Train and test ids
train_ids = train_raw['id']
test_ids = test_raw['id']

n_classes = 3

Reading file from: data\raw\train.csv
Reading file from: data\raw\test.csv


In [5]:
# X and y split
X = train_raw.drop(['id', 'irrigation_need'], axis=1)
y = train_raw['irrigation_need'].map(config.TARGET_MAP)
classes = np.unique(y)

# Test data
test_data = test_raw.drop('id', axis=1)

In [6]:
# Reduce memory size of data
X = reduce_mem_usage(X)
test_data = reduce_mem_usage(test_data)

Memory before: 354.06 MB
Memory after: 327.63 MB
Reduced by: 26.44 MB (7.5%)
Memory before: 151.74 MB
Memory after: 140.41 MB
Reduced by: 11.33 MB (7.5%)


In [7]:
# OOF
oof_proba_catgbm = read_csv_file(r'artifacts\oof\Catgbm_seed42_v5_oof_proba.csv')
oof_proba_catgbm = oof_proba_catgbm.iloc[:, 1:]

oof_proba_lr = read_csv_file(r'artifacts\oof\Logistic_seed42_v21_oof_proba.csv')
oof_proba_lr = oof_proba_lr.iloc[:, 1:]

# TEST_PROBA
test_proba_catgbm = read_csv_file(r'artifacts\test_proba\Catgbm_seed42_v5_test_proba.csv')
test_proba_catgbm = test_proba_catgbm.iloc[:, 1:]

test_proba_lr = read_csv_file(r'artifacts\test_proba\Logistic_seed42_v21_test_proba.csv')
test_proba_lr = test_proba_lr.iloc[:, 1:]

Reading file from: artifacts\oof\Catgbm_seed42_v5_oof_proba.csv
Reading file from: artifacts\oof\Logistic_seed42_v21_oof_proba.csv
Reading file from: artifacts\test_proba\Catgbm_seed42_v5_test_proba.csv
Reading file from: artifacts\test_proba\Logistic_seed42_v21_test_proba.csv


In [8]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

oof_catgbm = oof_proba_catgbm.values
oof_lr = oof_proba_lr.values

test_xgb = test_proba_catgbm.values
test_lr  = test_proba_lr.values

# quick sanity check
print("CATGBM OOF shape:", oof_catgbm.shape)
print("LR OOF shape:", oof_lr.shape)
print("CATGBM sample row:", oof_catgbm[0])
print("LR sample row:", oof_lr[0])

CATGBM OOF shape: (630000, 3)
LR OOF shape: (630000, 3)
CATGBM sample row: [9.92622840e-01 7.16004497e-03 2.17115244e-04]
LR sample row: [9.99851745e-01 1.48255211e-04 3.15014591e-11]


In [9]:
score_xgb = balanced_accuracy_score(y, np.argmax(oof_catgbm, axis=1))
score_lr  = balanced_accuracy_score(y, np.argmax(oof_lr,  axis=1))

print(f"CATGBM alone: {score_xgb:.6f}")
print(f"LR alone: {score_lr:.6f}")

CATGBM alone: 0.968407
LR alone: 0.976472


In [10]:
def objective(trial):

    w_catgbm = trial.suggest_float('w_catgbm', 0.0, 1.0)
    w_lr  = trial.suggest_float('w_lr',  0.0, 1.0)

    # normalize so weights sum to 1
    total = w_catgbm + w_lr
    w_catgbm /= total
    w_lr  /= total

    # blend raw probabilities
    blend = w_catgbm * oof_catgbm + w_lr * oof_lr

    # argmax to get class predictions
    preds = np.argmax(blend, axis=1)

    return balanced_accuracy_score(y, preds)


study = optuna.create_study(
    direction = 'maximize',
    sampler   = optuna.samplers.TPESampler(seed=42)
)

study.optimize(objective, n_trials=3000, show_progress_bar=True)

best  = study.best_params
total = best['w_catgbm'] + best['w_lr']

w_catgbm = best['w_catgbm'] / total
w_lr  = best['w_lr']  / total

print(f"\n{'='*40}")
print(f"XGB alone: {score_xgb:.6f}")
print(f"LR alone: {score_lr:.6f}")
print(f"Best blend: {study.best_value:.6f}")
print(f"Gain: +{study.best_value - max(score_xgb, score_lr):.6f}")
print(f"{'='*40}")
print(f"XGB weight: {w_catgbm:.4f} ({w_catgbm:.1%})")
print(f"LR weight: {w_lr:.4f} ({w_lr:.1%})")

  0%|          | 0/3000 [00:00<?, ?it/s]


XGB alone: 0.968407
LR alone: 0.976472
Best blend: 0.977031
Gain: +0.000559
XGB weight: 0.3037 (30.4%)
LR weight: 0.6963 (69.6%)


In [14]:
blend_oof = w_catgbm * oof_catgbm + w_lr * oof_lr
blend_test = w_catgbm * test_xgb + w_lr * test_lr

# -------- SAVE BLENDED FILES --------
blend_oof_df = pd.DataFrame(blend_oof, columns=[str(c) for c in classes])
blend_oof_df.insert(0, "id", train_ids)
save_csv_file(blend_oof_df, os.path.join(config.OOF_DIR, 'Logistic_seed42_v21_Catgbm_seed42_v5_blend_oof_proba.csv'))

blend_test_df = pd.DataFrame(blend_test, columns=[str(c) for c in classes])
blend_test_df.insert(0, "id", test_ids)
save_csv_file(blend_test_df, os.path.join(config.OOF_DIR, 'Logistic_seed42_v21_Catgbm_seed42_v5_blend_test_proba.csv'))

Saving file to: artifacts\oof\Logistic_seed42_v21_Catgbm_seed42_v5_blend_oof_proba.csv
Saving file to: artifacts\oof\Logistic_seed42_v21_Catgbm_seed42_v5_blend_test_proba.csv


### Threshold Tuning

In [19]:
blend_oof = read_csv_file('artifacts\oof\Logistic_seed42_v21_Catgbm_seed42_v5_blend_oof_proba.csv')
blend_oof = blend_oof.iloc[:, 1:]

blend_test_proba = read_csv_file('artifacts\oof\Logistic_seed42_v21_Catgbm_seed42_v5_blend_test_proba.csv')
blend_test_proba = blend_test_proba.iloc[:, 1:]

Reading file from: artifacts\oof\Logistic_seed42_v21_Catgbm_seed42_v5_blend_oof_proba.csv
Reading file from: artifacts\oof\Logistic_seed42_v21_Catgbm_seed42_v5_blend_test_proba.csv


In [21]:
n_classes = 3

def predict_with_thresholds(probs, thresholds):
    scaled = probs / (np.array(thresholds) + 1e-8)
    return np.argmax(scaled, axis=1)

def objective(trial):
    thresholds = [
        trial.suggest_float(f"thresh_{i}", 0.01, 0.99)
        for i in range(n_classes)
    ]
    preds = predict_with_thresholds(blend_oof, thresholds)
    return balanced_accuracy_score(y, preds)

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
study.optimize(objective, n_trials=5000, show_progress_bar=True)

best_thresholds = [study.best_params[f"thresh_{i}"] for i in range(n_classes)]

baseline  = balanced_accuracy_score(y, np.argmax(blend_oof, axis=1))
optimised = balanced_accuracy_score(y, predict_with_thresholds(blend_oof, best_thresholds))

abs_gain = optimised - baseline
pct_gain = (abs_gain / baseline) * 100 if baseline != 0 else 0

print(f"Baseline  BACC: {baseline}")
print(f"Optimised BACC: {optimised}")
print(f"Gain: +{abs_gain:.5f} ({pct_gain:.5f}%)")
print(f"Thresholds : class_0={best_thresholds[0]:.5f} | class_1={best_thresholds[1]:.5f} | class_2={best_thresholds[2]:.5f}")

  0%|          | 0/5000 [00:00<?, ?it/s]

Baseline  BACC: 0.9770308944908527
Optimised BACC: 0.9771616309761235
Gain: +0.00013 (0.01338%)
Thresholds : class_0=0.94436 | class_1=0.59215 | class_2=0.59064


In [22]:
out = predict_with_thresholds(blend_test_proba, best_thresholds)
out = pd.DataFrame({'id': test_ids, 'Irrigation_Need': out})
out.Irrigation_Need = out.Irrigation_Need.map(config.TARGET_MAP_INV)

save_path = os.path.join(config.SUBMISSIONS_DIR, 'Logistic_seed42_v21_Catgbm_seed42_v5_blend_thresh_tuned_submission.csv')
save_csv_file(out, save_path)

Saving file to: artifacts\submissions\Logistic_seed42_v21_Catgbm_seed42_v5_blend_thresh_tuned_submission.csv
